# Notebook 11: Zero-shot CLIP baseline and checkpoint names (reviewer item 2.3 and Methods details)

1. **Zero-shot CLIP** (no training): four CLIP models classify every BD-LW image from text prompts alone, with three prompt sets (English object name, English with templates, local name with English description). This shows whether general vision-language models already recognize these implements, i.e. whether a dedicated dataset is needed.
2. **Checkpoint names**: the default pretrained tag of every `timm` model used (for the Methods).

**Inputs (Add Input -> Your Work):** Stage 0 output only.
Settings: **GPU T4 x1 and Internet ON** (CLIP weights are downloaded). Time: about 15 to 25 min.

**Outputs (`/kaggle/working/clip_timing_cfg/`):** `clip_zero_shot.csv`, `clip_per_class_recall.csv`, `clip_confusion_best.csv`, `clip_predictions.csv`, `pretrained_cfg.csv`, and a zip.

In [1]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("timm") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm"])
import os, json, math, time, zipfile, warnings
from pathlib import Path
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import f1_score
import torch, torch.nn.functional as F_, timm
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ON_KAGGLE = Path("/kaggle/input").is_dir()
INPUT_ROOT = Path(os.environ.get("BD_INPUT", "/kaggle/input" if ON_KAGGLE else r"H:\BD Weapon Inputs"))
WORK = Path(os.environ.get("BD_WORK", "/kaggle/working" if ON_KAGGLE else r"H:\BD Weapon Analysis"))
MODELS_ALL = ["resnet50", "efficientnet_b0", "convnext_tiny", "vit_small_patch16_224"]
NAME = {"resnet50": "ResNet-50", "efficientnet_b0": "EfficientNet-B0", "convnext_tiny": "ConvNeXt-Tiny",
        "vit_small_patch16_224": "ViT-S/16"}
SEED, PAD_GRAY = 42, 128
print("device:", DEVICE, "| timm", timm.__version__, "| torch", torch.__version__)

def walk_find(base, predicate, skip=("standardized", "cutouts", "masks", "figures", "weights")):
    for dirpath, dirnames, filenames in os.walk(base, followlinks=True):
        hit = predicate(Path(dirpath), filenames)
        if hit:
            return hit
        dirnames[:] = [x for x in dirnames if x not in skip]
    return None

STAGE0 = walk_find(INPUT_ROOT, lambda d, f: d.parent if ("manifest.csv" in f and d.name == "metadata"
                                                        and d.parent.name == "stage0") else None)
RES = walk_find(INPUT_ROOT, lambda d, f: d if ("summary_all.csv" in f and d.name == "all"
                                              and d.parent.name == "results") else None)
assert STAGE0, "Stage 0 output not found (Add Input -> Your Work -> Stage 0 notebook)."
print("Stage 0   :", STAGE0); print("Notebook 2:", RES)

manifest = pd.read_csv(STAGE0 / "metadata" / "manifest.csv", dtype={"image_id": str})
CLASS_MAP = json.loads((STAGE0 / "metadata" / "class_map.json").read_text())
CLASSES = sorted(CLASS_MAP, key=CLASS_MAP.get)
manifest["y"] = manifest.label.map(CLASS_MAP)
test = manifest[manifest.split == "test"].reset_index(drop=True)
print(f"{len(manifest)} images, {len(test)} test images, classes: {CLASSES}")

def letterbox(img, res, pad):                    # identical to Notebook 2
    h, w = img.shape[:2]; s = res / max(h, w)
    nh, nw = max(1, round(h * s)), max(1, round(w * s))
    small = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC)
    out = np.full((res, res) + img.shape[2:], pad, img.dtype)
    t, l = (res - nh) // 2, (res - nw) // 2
    out[t:t + nh, l:l + nw] = small
    return out

def geometric(img, cfg, pad):
    if cfg["size"] == "stretch":
        return cv2.resize(img, (cfg["res"], cfg["res"]), interpolation=cv2.INTER_AREA)
    return letterbox(img, cfg["res"], pad)

def clahe_rgb(rgb):
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

def convert_color(rgb, color):
    if color == "gray":
        return np.repeat(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)[..., None], 3, axis=2)
    if color == "hsv":
        return cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV_FULL)
    if color == "lab":
        return cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    return rgb

summ = pd.read_csv(RES / "summary_all.csv").drop_duplicates(["model", "pipeline"], keep="last") if RES else None

def cfg_of(model, pipe="F"):
    s = summ[(summ.model == model) & (summ.pipeline == pipe)].iloc[0]
    return {"bg": s.cfg_bg, "size": s.cfg_size, "res": int(s.cfg_res), "color": s.cfg_color}

def folds_of(model, pipe="F"):
    return sorted(int(p.stem[4:]) for p in (RES / model / pipe / "weights").glob("fold*.pt"))

def load_net(model, fold, pipe="F"):
    cfg = cfg_of(model, pipe)
    kw = dict(pretrained=False, num_classes=len(CLASSES))
    if model.startswith("vit"):
        kw["img_size"] = cfg["res"]
    net = timm.create_model(model, **kw)
    sd = torch.load(RES / model / pipe / "weights" / f"fold{fold}.pt", map_location="cpu")
    net.load_state_dict({k: v.float() for k, v in sd.items()})
    nj = RES / model / pipe / "weights" / "norm.json"
    if nj.exists():
        n = json.loads(nj.read_text())[str(fold)]; mean, std = n["mean"], n["std"]
    else:
        mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    mean = torch.tensor(mean, dtype=torch.float32).view(1, 3, 1, 1)
    std = torch.tensor(std, dtype=torch.float32).view(1, 3, 1, 1)
    return net.eval().to(DEVICE), cfg, mean, std

def to_tensor(batch_uint8, color, mean, std):
    x = np.stack([convert_color(im, color) for im in batch_uint8])
    x = torch.from_numpy(x).permute(0, 3, 1, 2).float() / 255
    return (x - mean) / std

device: cuda | timm 1.0.30 | torch 2.11.0+cu130
Stage 0   : /kaggle/input/notebooks/arnicasarker/stage0-single-multi-3parts/stage0
Notebook 2: None
2950 images, 419 test images, classes: ['Churi', 'Dao', 'Khor', 'Kodal', 'Kural', 'Shabol']


In [2]:
if importlib.util.find_spec("open_clip") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch"])
import open_clip
OUT = WORK / "clip_timing_cfg"; OUT.mkdir(parents=True, exist_ok=True)
CLIP_MODELS = [("ViT-B-32", "laion2b_s34b_b79k"), ("ViT-B-16", "openai"),
               ("ViT-L-14", "openai"), ("ViT-L-14", "laion2b_s32b_b82k")]
EN = {"churi": "kitchen knife", "dao": "machete", "khor": "straight razor", "kodal": "hoe", "kural": "axe",
      "shabol": "iron crowbar"}
DESC = {"churi": "a knife", "dao": "a curved machete", "khor": "a folding straight razor", "kodal": "a hoe for digging",
        "kural": "an axe", "shabol": "a long iron crowbar"}
TEMPLATES = ["a photo of a {}.", "a close-up photo of a {}.", "a photo of a {} on the ground.",
             "a photo of a metal {}.", "a photo of a {}, a hand tool.", "a photo of several {}s."]
key = lambda c: c.lower().strip()
assert all(key(c) in EN for c in CLASSES), CLASSES
PROMPTS = {
    "name": {c: [f"a photo of a {EN[key(c)]}."] for c in CLASSES},
    "name+templates": {c: [t.format(EN[key(c)]) for t in TEMPLATES] for c in CLASSES},
    "local+description": {c: [f"a photo of a {key(c)}, {DESC[key(c)]} from Bangladesh.",
                              f"a photo of a Bangladeshi {key(c)}, which is {DESC[key(c)]}."] for c in CLASSES},
}
# letterbox to a square first so CLIP's centre crop cannot cut the weapon off
SQ = [Image.fromarray(letterbox(np.asarray(Image.open(STAGE0 / p).convert("RGB")), 448, PAD_GRAY))
      for p in manifest.path]
print(len(SQ), "images loaded")

res, preds = [], {}
y_all = manifest.y.to_numpy()
for arch, tag in CLIP_MODELS:
    t0 = time.time()
    model, _, prep = open_clip.create_model_and_transforms(arch, pretrained=tag, device=DEVICE)
    tok = open_clip.get_tokenizer(arch); model.eval()
    with torch.no_grad(), torch.autocast(device_type="cuda", enabled=(DEVICE == "cuda")):
        feats = []
        for i in range(0, len(SQ), 128):
            x = torch.stack([prep(im) for im in SQ[i:i + 128]]).to(DEVICE)
            feats.append(F_.normalize(model.encode_image(x).float(), dim=-1).cpu())
        feats = torch.cat(feats)
        for pset, pr in PROMPTS.items():
            W = []
            for c in CLASSES:
                t = F_.normalize(model.encode_text(tok(pr[c]).to(DEVICE)).float(), dim=-1).mean(0)
                W.append(F_.normalize(t, dim=0).cpu())
            p = (feats @ torch.stack(W).T).argmax(1).numpy()
            preds[(arch, tag, pset)] = p
            for part, msk in [("test", manifest.split == "test"),
                              ("test single", (manifest.split == "test") & (manifest.subset == "single")),
                              ("test multi", (manifest.split == "test") & (manifest.subset == "multi")),
                              ("all images", manifest.split.notna())]:
                msk = msk.to_numpy()
                res.append({"model": f"{arch} ({tag})", "prompts": pset, "images": part, "n": int(msk.sum()),
                            "macro_f1": f1_score(y_all[msk], p[msk], average="macro", labels=range(len(CLASSES)), zero_division=0),
                            "accuracy": (p[msk] == y_all[msk]).mean()})
    print(f"{arch} {tag}: {time.time() - t0:.0f} s")
    del model; torch.cuda.empty_cache() if DEVICE == "cuda" else None

clip = pd.DataFrame(res); clip.round(4).to_csv(OUT / "clip_zero_shot.csv", index=False)
display(clip[clip.images == "test"].pivot_table(index="model", columns="prompts", values="macro_f1").round(3))
best = clip[clip.images == "test"].sort_values("macro_f1").iloc[-1]
print("best zero-shot on the test set:", best.model, best.prompts, round(best.macro_f1, 3))
rec = []
for (arch, tag, pset), p in preds.items():
    for j, c in enumerate(CLASSES):
        m = (y_all == j) & (manifest.split == "test").to_numpy()
        rec.append({"model": f"{arch} ({tag})", "prompts": pset, "class": c, "recall": (p[m] == j).mean()})
pd.DataFrame(rec).pivot_table(index=["model", "prompts"], columns="class", values="recall").round(3
    ).to_csv(OUT / "clip_per_class_recall.csv")
bk = [k for k in preds if f"{k[0]} ({k[1]})" == best.model and k[2] == best.prompts][0]
tm = (manifest.split == "test").to_numpy()
cm = pd.crosstab(pd.Series(np.array(CLASSES)[y_all[tm]], name="true"),
                 pd.Series(np.array(CLASSES)[preds[bk][tm]], name="predicted")).reindex(index=CLASSES, columns=CLASSES, fill_value=0)
cm.to_csv(OUT / "clip_confusion_best.csv"); display(cm)
pd.DataFrame({"image_id": manifest.image_id, "label": manifest.label, "split": manifest.split, "subset": manifest.subset,
              **{f"{a}|{t}|{s}": np.array(CLASSES)[p] for (a, t, s), p in preds.items()}}
             ).to_csv(OUT / "clip_predictions.csv", index=False)

2950 images loaded


open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

ViT-B-32 laion2b_s34b_b79k: 27 s


open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

... (6 outputs left out)

ViT-L-14 laion2b_s32b_b82k: 63 s


prompts,local+description,name,name+templates
model,,,
ViT-B-16 (openai),0.506,0.316,0.317
ViT-B-32 (laion2b_s34b_b79k),0.549,0.448,0.437
ViT-L-14 (laion2b_s32b_b82k),0.667,0.550,0.560
ViT-L-14 (openai),0.707,0.586,0.633


best zero-shot on the test set: ViT-L-14 (openai) local+description 0.707


predicted,Churi,Dao,Khor,Kodal,Kural,Shabol
true,,,,,,
Churi,56,6,5,0,0,1
Dao,3,31,6,1,14,24
Khor,0,0,62,0,0,0
Kodal,0,1,1,39,23,3
Kural,0,1,1,5,50,6
Shabol,0,4,2,0,17,57


In [3]:
# default pretrained checkpoint of every timm model (what pretrained=True loads in this timm version)
rows = []
for name in ["resnet50", "efficientnet_b0", "convnext_tiny", "vit_small_patch16_224", "vit_small_patch14_dinov2"]:
    c = timm.create_model(name, pretrained=False).pretrained_cfg
    rows.append({"model": name, "timm_version": timm.__version__, "tag": c.get("tag"),
                 "checkpoint": f"{c.get('architecture', name)}.{c.get('tag')}" if c.get("tag") else name,
                 "hf_hub_id": c.get("hf_hub_id"), "input_size": c.get("input_size"), "mean": c.get("mean"),
                 "std": c.get("std"), "license": c.get("license")})
cfgs = pd.DataFrame(rows); cfgs.to_csv(OUT / "pretrained_cfg.csv", index=False); display(cfgs)
# if Notebook 2 saved its own timm version or checkpoint names, show them
for f in (RES.rglob("*.json") if RES else []):
    if f.name in ("config.json", "env.json", "versions.json", "run_config.json"):
        print(f, f.read_text()[:400])
with zipfile.ZipFile(WORK / "clip_timing_cfg.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in OUT.iterdir():
        z.write(f, f"{OUT.name}/{f.name}")
print("Done:", WORK / "clip_timing_cfg.zip")

,model,timm_version,tag,checkpoint,hf_hub_id,input_size,mean,std,license
0,resnet50,1.0.30,a1_in1k,resnet50.a1_in1k,timm/resnet50.a1_in1k,"(3, 224, 224)","(0.485, 0.456, 0.406)","(0.229, 0.224, 0.225)",apache-2.0
1,efficientnet_b0,1.0.30,ra_in1k,efficientnet_b0.ra_in1k,timm/efficientnet_b0.ra_in1k,"(3, 224, 224)","(0.485, 0.456, 0.406)","(0.229, 0.224, 0.225)",apache-2.0
2,convnext_tiny,1.0.30,in12k_ft_in1k,convnext_tiny.in12k_ft_in1k,timm/convnext_tiny.in12k_ft_in1k,"(3, 224, 224)","(0.485, 0.456, 0.406)","(0.229, 0.224, 0.225)",apache-2.0
3,vit_small_patch16_224,1.0.30,augreg_in21k_ft_in1k,vit_small_patch16_224.augreg_in21k_ft_in1k,timm/vit_small_patch16_224.augreg_in21k_ft_in1k,"(3, 224, 224)","(0.5, 0.5, 0.5)","(0.5, 0.5, 0.5)",apache-2.0
4,vit_small_patch14_dinov2,1.0.30,lvd142m,vit_small_patch14_dinov2.lvd142m,timm/vit_small_patch14_dinov2.lvd142m,"(3, 518, 518)","(0.485, 0.456, 0.406)","(0.229, 0.224, 0.225)",apache-2.0


Done: /kaggle/working/clip_timing_cfg.zip
